In [0]:
from config_loader import ConfigLoader
import os

os.environ["DATABRICKS_WORKSPACE_ID"] = "4142847529395360"
os.environ["DATABRICKS_FRAMEWORK_PATH_ENV"] = "/Workspace/Shared/config/bundle.yml"

env = ConfigLoader()


In [0]:
from pyspark.sql import SparkSession

class EnvironmentManager:
    """
    Clase encargada de preparar el entorno Unity Catalog y almacenamiento:
      - Validar o crear Catalog
      - Validar o crear Schema
      - Validar o crear Storage Credential
      - Validar o crear External Location
    """

    def __init__(self, spark: SparkSession, config: dict):
        self.spark = spark
        self.config = config
        self.cloud = config.get("cloud", "azure").lower()  # azure / aws / gcp
        self.catalog = config.get("catalog")
        self.schema = config.get("schema")
        self.external_location = config.get("external_location", {})
        self._is_ready = False

    # --------------------------------------------------------------------------
    # AUXILIARES
    # --------------------------------------------------------------------------

    def _execute_safe_sql(self, sql_text: str):
        try:
            self.spark.sql(sql_text)
            return True
        except Exception as e:
            print(f"⚠️ Error ejecutando SQL: {sql_text}\n→ {e}")
            return False

    def _object_exists(self, show_cmd: str, field: str, name: str) -> bool:
        """Verifica si un objeto (catalog/schema/etc) existe."""
        try:
            objs = [row[field] for row in self.spark.sql(show_cmd).collect()]
            return name in objs
        except Exception:
            return False

    # --------------------------------------------------------------------------
    # CATALOG
    # --------------------------------------------------------------------------

    def ensure_catalog(self):
        if not self._object_exists("SHOW CATALOGS", "catalog", self.catalog):
            print(f"🧱 Creando catálogo: {self.catalog}")
            self._execute_safe_sql(f"CREATE CATALOG IF NOT EXISTS {self.catalog}")
        else:
            print(f"ℹ️ Catálogo '{self.catalog}' ya existe.")

    # --------------------------------------------------------------------------
    # SCHEMA
    # --------------------------------------------------------------------------

    def ensure_schema(self):
        print(f"📂 Verificando schema: {self.catalog}.{self.schema}")
        self._execute_safe_sql(f"CREATE SCHEMA IF NOT EXISTS {self.catalog}.{self.schema}")

    # --------------------------------------------------------------------------
    # STORAGE CREDENTIAL
    # --------------------------------------------------------------------------

    def ensure_storage_credential(self):
        cred = self.external_location.get("credential")
        if not cred:
            print("⚠️ No se definió ninguna credencial de almacenamiento en la configuración.")
            return

        exists = self._object_exists("SHOW STORAGE CREDENTIALS", "name", cred)
        if exists:
            print(f"ℹ️ Storage credential '{cred}' ya existe.")
            return

        print(f"🪪 Creando storage credential '{cred}'...")

        if self.cloud == "azure":
            sql = f"""
            CREATE STORAGE CREDENTIAL {cred}
            WITH AZURE_MANAGED_IDENTITY;
            """
        elif self.cloud == "aws":
            iam_role = self.external_location.get("iam_role")
            if not iam_role:
                raise ValueError("Falta 'iam_role' en configuración para AWS.")
            sql = f"""
            CREATE STORAGE CREDENTIAL {cred}
            WITH IAM_ROLE='{iam_role}';
            """
        elif self.cloud == "gcp":
            service_acc = self.external_location.get("service_account")
            if not service_acc:
                raise ValueError("Falta 'service_account' en configuración para GCP.")
            sql = f"""
            CREATE STORAGE CREDENTIAL {cred}
            WITH GCP_SERVICE_ACCOUNT '{service_acc}';
            """
        else:
            raise ValueError(f"Nube '{self.cloud}' no soportada.")

        self._execute_safe_sql(sql)
        print(f"✅ Storage credential '{cred}' creada o validada.")

    # --------------------------------------------------------------------------
    # EXTERNAL LOCATION
    # --------------------------------------------------------------------------

    def ensure_external_location(self):
        loc = self.external_location.get("name")
        url = self.external_location.get("url")
        cred = self.external_location.get("credential")

        if not all([loc, url, cred]):
            print("⚠️ Parámetros incompletos en external_location (name, url, credential).")
            return

        exists = self._object_exists("SHOW EXTERNAL LOCATIONS", "name", loc)
        if exists:
            print(f"ℹ️ External location '{loc}' ya existe.")
            return

        print(f"🌐 Creando external location '{loc}'...")
        sql = f"""
        CREATE EXTERNAL LOCATION {loc}
        URL '{url}'
        WITH (STORAGE CREDENTIAL `{cred}`);
        """
        self._execute_safe_sql(sql)
        print(f"✅ External location '{loc}' creada o validada.")

    # --------------------------------------------------------------------------
    # FULL SETUP
    # --------------------------------------------------------------------------

    def setup_environment(self):
        print("🔧 Iniciando configuración del entorno...")
        self.ensure_catalog()
        self.ensure_schema()
        self.ensure_storage_credential()
        self.ensure_external_location()
        self._is_ready = True
        print("✅ Entorno Unity Catalog configurado correctamente.")

    # --------------------------------------------------------------------------
    # STATUS
    # --------------------------------------------------------------------------

    @property
    def is_ready(self):
        return self._is_ready


In [0]:
from config_loader import ConfigLoader
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# 1️⃣ Cargar configuración automáticamente
cfg_loader = ConfigLoader()
config = cfg_loader.get_config()

# 2️⃣ Inicializar EnvironmentManager
env = EnvironmentManager(spark, config)

# 3️⃣ Ejecutar configuración del entorno
env.setup_environment()
